# Pandas для игрового аналитика: пропуски, группировки, джойны
Работа с реальными данными A/B теста на выручке и retention.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

## Раздел 1: Что в руках
Загружаем три таблицы и проверяем гранулярность: каждая строка — это действительно один игрок?

In [2]:
user_level = pd.read_csv('../data/user_level.csv')
daily = pd.read_csv('../data/daily.csv')
retention = pd.read_csv('../data/retention.csv')

print(f"user_level shape: {user_level.shape}")
print(f"daily shape: {daily.shape}")
print(f"retention shape: {retention.shape}")

user_level shape: (100000, 8)
daily shape: (118, 6)
retention shape: (5, 9)


In [3]:
# Гранулярность user_level: количество уникальных пользователей должно равняться числу строк
print(f"Уникальных user_id: {user_level['user_id'].nunique()}")
print(f"Строк в таблице: {len(user_level)}")
print(f"Гранулярность корректна: {user_level['user_id'].nunique() == len(user_level)}")

Уникальных user_id: 100000
Строк в таблице: 100000
Гранулярность корректна: True


In [4]:
print("\nuser_level info:")
print(user_level.info())
print("\nuser_level dtypes:")
print(user_level.dtypes)


user_level info:
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   user_id             100000 non-null  int64  
 1   group               100000 non-null  str    
 2   total_revenue       100000 non-null  float64
 3   total_sessions      100000 non-null  int64  
 4   total_time          100000 non-null  float64
 5   total_transactions  100000 non-null  int64  
 6   active_days         100000 non-null  int64  
 7   is_payer            100000 non-null  bool   
dtypes: bool(1), float64(2), int64(4), str(1)
memory usage: 5.4 MB
None

user_level dtypes:
user_id                 int64
group                     str
total_revenue         float64
total_sessions          int64
total_time            float64
total_transactions      int64
active_days             int64
is_payer                 bool
dtype: object


In [5]:
print("\nРаспределение по группам (нормализованное):")
print(user_level['group'].value_counts(normalize=True))
print("\nОписательная статистика по выручке и сессиям:")
print(user_level[['total_revenue', 'total_sessions', 'total_time']].describe())


Распределение по группам (нормализованное):
group
B    0.50103
A    0.49897
Name: proportion, dtype: float64

Описательная статистика по выручке и сессиям:
       total_revenue  total_sessions     total_time
count  100000.000000   100000.000000  100000.000000
mean        5.703866       35.277950    1139.796395
std        16.500821       14.487675     447.582932
min         0.000000        0.000000      19.700000
25%         0.000000       30.000000    1007.200000
50%         0.000000       38.000000    1237.800000
75%         0.000000       45.000000    1431.100000
max       235.640000       80.000000    2664.500000


## Раздел 2: Пропуски
В исходных данных пропусков нет. Создадим их естественным путём через merge и разберём стратегии заполнения.

In [6]:
print("Пропуски в исходных таблицах:")
print(f"user_level: {user_level.isna().sum().sum()}")
print(f"daily: {daily.isna().sum().sum()}")
print(f"retention: {retention.isna().sum().sum()}")
print("Вывод: данные чистые на входе.")

Пропуски в исходных таблицах:
user_level: 0
daily: 0
retention: 0
Вывод: данные чистые на входе.


In [7]:
# Пропусков в исходных файлах нет, поэтому создаём их естественным путём:
# приклеиваем к полной таблице срез только по группе A через left join по составному ключу.
# У строк группы B пары не найдётся, и в приклеенных колонках появятся NaN.
daily_a = (daily[daily['group'] == 'A']
           [['date', 'group', 'revenue', 'users']]
           .rename(columns={'revenue': 'revenue_a', 'users': 'users_a'}))

daily_gaps = daily.merge(daily_a, on=['date', 'group'], how='left')

print(f"Строк: {len(daily_gaps)}")
print("Пропуски по колонкам:")
print(daily_gaps.isna().sum())

Строк: 118
Пропуски по колонкам:
date             0
group            0
revenue          0
users            0
transactions     0
arpdau           0
revenue_a       59
users_a         59
dtype: int64


Пропуск в `revenue_a` означает «этой строки не было в правой таблице», то есть ноль выручки
по определению. Такой пропуск заполнять нулём корректно. А вот пропуск в дате или в
идентификаторе означал бы «мы не знаем» — заполнять его нельзя, такие строки убирают.

In [8]:
# fillna: заполняем там, где ноль осмыслен
daily_filled = daily_gaps.copy()
daily_filled['revenue_a'] = daily_filled['revenue_a'].fillna(0)

print(f"NaN в revenue_a до fillna:  {daily_gaps['revenue_a'].isna().sum()}")
print(f"NaN в revenue_a после fillna: {daily_filled['revenue_a'].isna().sum()}")
print(f"Сумма revenue_a после заполнения: {daily_filled['revenue_a'].sum():.2f}")

NaN в revenue_a до fillna:  59
NaN в revenue_a после fillna: 0
Сумма revenue_a после заполнения: 272053.61


### dropna против drop_duplicates — разные задачи
`dropna` убирает строки, где **нет значения**. `drop_duplicates` убирает строки, которые
**повторяют** другие строки. Их легко перепутать на слух, но они решают противоположные
проблемы: первый борется с неполнотой данных, второй — с их задвоением.

In [9]:
# dropna: выбрасываем строки, где нет значения в указанной колонке
daily_no_na = daily_gaps.dropna(subset=['revenue_a'])

print(f"dropna(subset=['revenue_a']): {len(daily_no_na)} строк из {len(daily_gaps)}")
print(f"Выброшено: {len(daily_gaps) - len(daily_no_na)} строк — это ровно группа B")
print(f"Осталось групп: {daily_no_na['group'].unique().tolist()}")

dropna(subset=['revenue_a']): 59 строк из 118
Выброшено: 59 строк — это ровно группа B
Осталось групп: ['A']


In [10]:
# drop_duplicates: решает другую задачу — задвоение строк
print(f"Полных дубликатов в user_level: {user_level.duplicated().sum()}")
print(f"Дубликатов по user_id:          {user_level.duplicated(subset=['user_id']).sum()}")
print(f"drop_duplicates() оставит:      {len(user_level.drop_duplicates())} строк из {len(user_level)}")

Полных дубликатов в user_level: 0
Дубликатов по user_id:          0
drop_duplicates() оставит:      100000 строк из 100000


**КЛЮЧЕВАЯ РАЗНИЦА: `dropna` vs `drop_duplicates`**

- `dropna()` удаляет строки, содержащие пропущенные значения (NaN)
- `drop_duplicates()` удаляет полностью идентичные строки (дубликаты)

Они работают с разными проблемами: первый — с пропусками, второй — с повторениями. На вопрос «как удалить строки с пропусками» правильный ответ всегда `dropna`, а не `drop_duplicates`.

In [11]:
# Проверка дубликатов в user_level
print(f"Дубликатов в user_level (полных совпадений): {user_level.duplicated().sum()}")
print(f"Дубликатов по user_id: {user_level.duplicated(subset=['user_id']).sum()}")

Дубликатов в user_level (полных совпадений): 0
Дубликатов по user_id: 0


In [12]:
# Удаление дубликатов (в нашем случае их нет, но вот синтаксис)
user_dedup = user_level.drop_duplicates(subset=['user_id'])
print(f"Строк после drop_duplicates: {len(user_dedup)} (было {len(user_level)})")

Строк после drop_duplicates: 100000 (было 100000)


## Раздел 3: Отбор строк
Найдём платящих игроков с выручкой выше медианы в своей группе.

In [13]:
# Способ 1: булева маска с loc
median_revenue = user_level['total_revenue'].median()
payers_above_median = user_level.loc[
    (user_level['is_payer'] == True) & (user_level['total_revenue'] > median_revenue)
]
print(f"Платящих с выручкой > медианы ({median_revenue:.2f}): {len(payers_above_median)}")
print(f"В том числе группа A: {len(payers_above_median[payers_above_median['group']=='A'])}")

Платящих с выручкой > медианы (0.00): 16471
В том числе группа A: 8280


In [14]:
# Способ 2: query для более читаемых условий
payers_query = user_level.query('is_payer == True and total_revenue > @median_revenue')
print(f"query результат совпадает: {len(payers_query) == len(payers_above_median)}")

query результат совпадает: True


In [15]:
# Средние показатели для платящих с высокой выручкой
print("\nСредние показатели платящих выше медианы:")
print(payers_above_median.groupby('group')[['total_revenue', 'total_sessions']].mean())


Средние показатели платящих выше медианы:
       total_revenue  total_sessions
group                               
A          32.856716       38.820048
B          36.422047       41.744354


## Раздел 4: Группировки и трансформации
Посчитаем статистику по группам, затем трансформируем каждого игрока отношением его выручки к сумме группы.

In [16]:
# Агрегация: сводим 100k игроков в 2 строки по группам
group_stats = user_level.groupby('group').agg(
    players_count=('user_id', 'nunique'),
    total_revenue=('total_revenue', 'sum'),
    payers_count=('is_payer', 'sum'),
    avg_session_length=('total_time', 'mean')
).round(2)
print("Статистика по группам:")
print(group_stats)

Статистика по группам:
       players_count  total_revenue  payers_count  avg_session_length
group                                                                
A              49897      272053.61          8280             1088.74
B              50103      298332.99          8191             1190.65


In [17]:
# Трансформация: каждому игроку добавляем долю его выручки от группы
# transform возвращает 100k строк, как исходная таблица
user_level['group_revenue_total'] = (
    user_level.groupby('group')['total_revenue'].transform('sum')
)
user_level['revenue_pct'] = (
    user_level['total_revenue'] / user_level['group_revenue_total'] * 100
).round(4)

print(f"\nФорма user_level после transform: {user_level.shape}")
print("Примеры доли выручки по группам:")
print(user_level[['user_id', 'group', 'total_revenue', 'revenue_pct']].head(3))


Форма user_level после transform: (100000, 10)
Примеры доли выручки по группам:
   user_id group  total_revenue  revenue_pct
0        1     A            0.0          0.0
1        2     B            0.0          0.0
2        3     B            0.0          0.0


In [18]:
# Построчный apply(axis=1): для каждой строки вызывается функция Python.
# Считаем выручку на сессию — задача, которую новичок обычно решает именно так.
def revenue_per_session(row):
    if row['total_sessions'] > 0:
        return row['total_revenue'] / row['total_sessions']
    return 0.0

%timeit user_level.apply(revenue_per_session, axis=1)

628 ms ± 21.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [19]:
# Та же задача векторно: деление колонки на колонку, ноль обрабатывается через np.where.
safe_sessions = user_level['total_sessions'].replace(0, np.nan)

%timeit np.where(user_level['total_sessions'] > 0, user_level['total_revenue'] / safe_sessions, 0.0)

443 μs ± 11.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [20]:
# Проверяем, что оба способа дают одинаковый результат
by_apply = user_level.apply(revenue_per_session, axis=1)
by_vector = np.where(user_level['total_sessions'] > 0,
                     user_level['total_revenue'] / safe_sessions, 0.0)

print(f"Результаты совпадают: {np.allclose(by_apply.values, by_vector)}")
print(f"Пример значений: {by_vector[:3].round(4)}")

Результаты совпадают: True
Пример значений: [0. 0. 0.]


Разница во времени видна в выводе выше: построчный `apply` медленнее векторной операции
на порядки, потому что вызывает функцию Python для каждой из ста тысяч строк.
Поэтому `apply` в продакшене избегают и оставляют для случаев, где векторного аналога нет.

## Раздел 5: Джойны
Смерживаем статистику по группам к каждому игроку, проверяем корректность через validate.

In [21]:
# Корректный merge: много игроков на одну строку группы (many_to_one)
group_metrics = user_level.groupby('group').agg(
    group_arpu=('total_revenue', 'mean'),
    group_engagement=('total_sessions', 'mean')
).reset_index()

print("Статистика для merge:")
print(group_metrics)

Статистика для merge:
  group  group_arpu  group_engagement
0     A    5.452304         34.124336
1     B    5.954394         36.426821


In [22]:
# Смерживаем с валидацией
user_enriched = user_level.merge(
    group_metrics,
    on='group',
    how='left',
    validate='many_to_one'
)

print(f"\nПосле merge строк: {len(user_enriched)} (было {len(user_level)})")
print("Валидация прошла: связь действительно many-to-one")
print(user_enriched[['user_id', 'group', 'total_revenue', 'group_arpu']].head(3))


После merge строк: 100000 (было 100000)
Валидация прошла: связь действительно many-to-one
   user_id group  total_revenue  group_arpu
0        1     A            0.0    5.452304
1        2     B            0.0    5.954394
2        3     B            0.0    5.954394


In [23]:
# Намеренно создаём ошибку: дублируем одну группу в метриках
bad_metrics = pd.concat([
    group_metrics,
    group_metrics[group_metrics['group'] == 'A']
], ignore_index=True)

print(f"\nПлохие метрики (группа A повторена):\n{bad_metrics}")


Плохие метрики (группа A повторена):
  group  group_arpu  group_engagement
0     A    5.452304         34.124336
1     B    5.954394         36.426821
2     A    5.452304         34.124336


In [24]:
# Попытка merge с нарушением связи many_to_one
try:
    bad_merge = user_level.merge(
        bad_metrics,
        on='group',
        how='left',
        validate='many_to_one'
    )
except Exception as e:
    print(f"Ошибка валидации (как и ожидается):\n{type(e).__name__}: {str(e)[:200]}")

Ошибка валидации (как и ожидается):
MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 group
    A ...


## Раздел 6: Воспроизведение результата
Пересчитаем метрики retention, включая p-value z-теста для дня 1.

In [25]:
print("Данные retention из файла:")
print(retention)

Данные retention из файла:
   day      A_pct      B_pct      lift             p  A_ret    A_n  B_ret  \
0    1  41.254985  39.837934 -3.434861  5.032345e-06  20585  49897  19960   
1    3  36.545283  34.520887 -5.539419  2.264990e-11  18235  49897  17296   
2    7  32.144217  30.774604 -4.260837  3.106984e-06  16039  49897  15419   
3   14  29.739263  29.656907 -0.276927  7.756579e-01  14839  49897  14859   
4   28  29.625028  29.624973 -0.000186  9.998480e-01  14782  49897  14843   

     B_n  
0  50103  
1  50103  
2  50103  
3  50103  
4  50103  


In [26]:
# Пересчитаем процент retention для группы A в день 1
retention['A_pct_calc'] = (retention['A_ret'] / retention['A_n'] * 100).round(4)
retention['B_pct_calc'] = (retention['B_ret'] / retention['B_n'] * 100).round(4)

day1 = retention[retention['day'] == 1].iloc[0]
print(f"\nДень 1, группа A:")
print(f"  Возвращено: {day1['A_ret']} из {day1['A_n']}")
print(f"  Пересчитано: {day1['A_pct_calc']:.4f}%")
print(f"  Публиковано: {day1['A_pct']:.4f}%")
print(f"  Совпадение: {abs(day1['A_pct_calc'] - day1['A_pct']) < 0.0001}")


День 1, группа A:
  Возвращено: 20585.0 из 49897.0
  Пересчитано: 41.2550%
  Публиковано: 41.2550%
  Совпадение: True


In [27]:
# Z-тест двух пропорций для дня 1
p_a = day1['A_ret'] / day1['A_n']
p_b = day1['B_ret'] / day1['B_n']
n_a = day1['A_n']
n_b = day1['B_n']

# Объединённая доля (under null hypothesis: обе группы одинаковы)
p_pool = (day1['A_ret'] + day1['B_ret']) / (day1['A_n'] + day1['B_n'])

# Стандартная ошибка разницы
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b))

# Z-статистика
z_stat = (p_a - p_b) / se

# P-value (двусторонний тест)
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print(f"Z-тест для дня 1:")
print(f"  Группа A: {day1['A_ret']}/{day1['A_n']} = {p_a:.6f}")
print(f"  Группа B: {day1['B_ret']}/{day1['B_n']} = {p_b:.6f}")
print(f"  Z-статистика: {z_stat:.4f}")
print(f"  P-value (расчёт): {p_value:.3e}")
print(f"  P-value (файл): {day1['p']:.3e}")
print(f"  Совпадение: {abs(np.log10(p_value) - np.log10(day1['p'])) < 0.1}")

Z-тест для дня 1:
  Группа A: 20585.0/49897.0 = 0.412550
  Группа B: 19960.0/50103.0 = 0.398379
  Z-статистика: 4.5634
  P-value (расчёт): 5.032e-06
  P-value (файл): 5.032e-06
  Совпадение: True


In [28]:
# Сравнение retention в день 1 и день 28
day28 = retention[retention['day'] == 28].iloc[0]
print(f"\nДень 1: A={day1['A_pct']:.2f}% vs B={day1['B_pct']:.2f}% (p={day1['p']:.3e}) — значимо разные")
print(f"День 28: A={day28['A_pct']:.2f}% vs B={day28['B_pct']:.2f}% (p={day28['p']:.4f}) — различий нет")


День 1: A=41.25% vs B=39.84% (p=5.032e-06) — значимо разные
День 28: A=29.63% vs B=29.62% (p=0.9998) — различий нет


**Вывод:** Retention группы B в день 1 значимо ниже группы A (41.26% vs 39.84%, p=5e-06).
Однако к дню 28 доля вернувшихся выравнивается (29.63% в обеих группах, p=0.9998).
Это означает, что обе группы теряют примерно одинаковую долю пользователей, но в группе B уход начинается раньше.